# Phase 3 - Data Preparation

#### Prepares the raw Telco dataset for ML modeling (Phases 4 and 5) by:

- Loading the `telco_churn_raw` CSV from `data\raw\telco_churn_raw.csv`,
- Inspecting and verifying data quality findings from data profiling (Phase 1) and EDA (Phase 2) programmatically, 
- Renaming columns to match project schema,
- Converting the `total_charges` entries from string to numeric,
- Removing observations containing blank `total_charges` entries post-conversion,
- Removing identifier and redundant features that were flagged during EDA,
- Encoding target variable `churn` using `LabelEncoder`,
- Creating training and test data through a $80/20$ split with reproducible stratification,
- One-hot encoding categorical features and standardises numeric features using a column transformer,
- Exporting the processed training and test `X_train_processed`, `X_test_processed`, `y_train`, and `y_test` CSVs to `data/processed/` for subsequent stages in the data pipeline, and
- Establishing and exporting a relational database `telco_churn.db` populated with churn dataframe (before feature removal, encoding, and scaling) for querying in Phase 7. 

#### Data source: [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

------------


##### $\textbf{Setup}$

Imports necessary libraries: `numpy`, `pandas`, `ColumnTransformer`, `LabelEncoder`, `StandardScaler`, `OneHotEncoder`, `train_test_split` for data manipulation, preprocessing and preparation operations; `sqlite3` for database creation; `warnings` for potential issue alerts; `os`, `Path` for directory navigation: 

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import os
from pathlib import Path   
import warnings

import sqlite3

Navigates from notebook's location to project root, i.e. since notebooks live in `notebooks/`, goes up one level to reach project root:

In [2]:
dir_nb = Path.cwd()
project_root = dir_nb.parent  
os.chdir(project_root)           # os.chdir accepts a Path object directly       

print(Path.cwd())                # prints project root directory               

C:\Users\Wits-User\Desktop\PROJECTS1\telco-churn-prediction


Ignores all potential warnings:

In [3]:
warnings.filterwarnings("ignore")

##### $\textbf{Data loading and inspection}$

Loads data from file:

In [4]:
churn_raw_df = pd.read_csv("data/raw/telco_churn_raw.csv")

print("Raw Telco churn prediction dataset read into dataframe!")

Raw Telco churn prediction dataset read into dataframe!


Verifies dimensions and additional structural metadata of `churn_raw_df` to programmatically confirm observations made during profiling and EDA:

In [5]:
churn_raw_df.shape  # expected: (7043, 21)

(7043, 21)

In [6]:
churn_raw_df.info()  # expected: dtypes = 1 float64, 2 int64, 18 str; 7043 entries non-null across all variables

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
print("Duplicates:",       churn_raw_df.duplicated().sum())       # expected: 0
print("Zero entries:\n",  (churn_raw_df == 0).sum(), sep = "")    # expected: all 0 except SeniorCitizen (5901) and tenure (11)
print("Empty strings:\n",  churn_raw_df.eq(" ").sum(), sep = "")  # expected: all 0 except TotalCharges (11) 

Duplicates: 0
Zero entries:
customerID             0
gender                 0
SeniorCitizen       5901
Partner                0
Dependents             0
tenure                11
PhoneService           0
MultipleLines          0
InternetService        0
OnlineSecurity         0
OnlineBackup           0
DeviceProtection       0
TechSupport            0
StreamingTV            0
StreamingMovies        0
Contract               0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges         0
TotalCharges           0
Churn                  0
dtype: int64
Empty strings:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
Month

##### $\textbf{Data preprocessing}$

Renames most column labels to match project schema. This step is executed without capital standardisation for better readability. `gender` and `tenure` are excluded from the renaming process, as their column labels were confirmed to be minuscule (lack of capital standardisation) during data profiling and by programmatic verification:

In [8]:
# assigns dataset to new variable, as it's no longer raw:
churn_df = churn_raw_df      

# extracts column names and index positions (column order confirmed via churn_raw_df.info()):
column_labels = churn_df.columns

# 'gender' and 'tenure' already minuscule:
churn_raw_df.rename(columns = {
    column_labels[0]  : 'customer_id',
    column_labels[2]  : 'senior_citizen',
    column_labels[3]  : 'partner',
    column_labels[4]  : 'dependents',
    column_labels[6]  : 'phone_service',
    column_labels[7]  : 'multiple_lines',
    column_labels[8]  : 'internet_service',
    column_labels[9]  : 'online_security',
    column_labels[10] : 'online_backup',
    column_labels[11] : 'device_protection',
    column_labels[12] : 'tech_support',
    column_labels[13] : 'streaming_tv',
    column_labels[14] : 'streaming_movies',
    column_labels[15] : 'contract',
    column_labels[16] : 'paperless_billing',
    column_labels[17] : 'payment_method',
    column_labels[18] : 'monthly_charges',
    column_labels[19] : 'total_charges',
    column_labels[20] : 'churn',
    }, inplace = True)

# confirms renamed column labels:
print(churn_df.columns.tolist())

['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn']


Programmatic verification confirms the presence of $5901$ zero entries for `senior_citizen` due to the feature already being one-hot encoded for `Yes` ($= 1$) and `No`($= 0$) in the raw dataset. According to the observations made during data profiling, there should be four numeric data-type columns:`senior_citizen`, `tenure`, `monthly_charges`, and `total_charges`. However, as established during EDA and through programmatic verification, `total_charges` is described as an `object` data type from the structural metadata. Furthermore, $11$ `total_charges` empty strings were identified during data profiling, confirmed during EDA, and verified programmatically. As such, `total_charges` entries require a conversion from string to numeric, with an approach to dealing with the empty strings.

Converts valid `total_charges` feature entries to numeric, with invalid strings transformed into `NaN`:

In [9]:
churn_df["total_charges"] = pd.to_numeric(churn_df["total_charges"], errors = "coerce")

# confirms numeric conversion and resulting transformations to NaN:
print("total_charges entries after conversion:",           churn_df["total_charges"].dtypes)         # expected: float64
print("Invalid total_charges entries transformed to NaN:", churn_df["total_charges"].isnull().sum()) # expected: 11

total_charges entries after conversion: float64
Invalid total_charges entries transformed to NaN: 11


When investigating the row entries containing the $11$ `total_charges` blank entries (formerly empty strings) during EDA, it was observed that all $11$ entries have a tenure value of $0$ (confirmed programmatically) and a churn class of `0`:   

In [10]:
null_df = churn_df[["total_charges", "tenure", "churn"]]

null_entries = null_df[null_df["total_charges"].isnull()]
null_entries

,total_charges,tenure,churn
488,NaN,0,No
753,NaN,0,No
936,NaN,0,No
1082,NaN,0,No
1340,NaN,0,No
3331,NaN,0,No
3826,NaN,0,No
4380,NaN,0,No
5218,NaN,0,No
6670,NaN,0,No


As established during EDA, the blank `total_charges` values and corresponding `tenure` and `churn` entries make coherent sense in context: customers with no tenure are new sign ups with their first billing cycle being incomplete, thus accruing no total charge as yet. Additionally, given that these customers are new sign ups, they would automatically be retained, making them non-churners. These observations therefore support the **exclusion** of the $11$ rows containing the blank entries, with the finalised decision based on the following:

- Given that the objective for modelling churn prediction is directed at customers that have at least some history (`tenure`$> 0$), the $11$ new sign ups (with their first incomplete billing cycle) fall outside the target population. 
- In EDA, it was established that the churn class split is $\sim 26/74$, with non-churners being the majority target class. Given this heavy class imbalance, the impact that removing row entries of $11$ non-churners has on shifting the imbalance (and churn prediction) is negligible.

Modifies `churn_df` by removing the $11$ rows containing the blank `total_charges` entries:

In [11]:
churn_df.dropna(inplace = True, ignore_index = True)

# confirms total rows and lack of blank entries post-exclusion:
print("Total row entries after exclusion:",         churn_df.shape[0])                         # expected: 7032
print("Null total_charges values after removal:",   churn_df["total_charges"].isnull().sum())  # expected: 0
print("Customers with zero tenure after removal:", (churn_df["tenure"] == 0).sum())            # expected: 0

# creates copy of dataset for database creation: 
db_churn_df = churn_df.copy()

Total row entries after exclusion: 7032
Null total_charges values after removal: 0
Customers with zero tenure after removal: 0


During EDA, a total of $7$ redundant features were flagged as potential candidates for removal, with the flagging criteria basis being individual univariate churn discrimination and correlation/associative strength across different feature pairs. Majority of these features comprised deterministic/near-deterministic pairs with underlying structural mechanisms. This criteria supports the **removal** of all $7$ features, with the finalised decision based on the following:

- `gender`: weak univariate churn discrimination; negligible associations across all feature pairs via Cramér's V and eta-squared magnitudes. 
- `phone_service`: weak univariate churn discrimination; fully deterministic/structural relationship with `multiple_lines` via Cramér's V magnitude.
- `multiple_lines`: weak univariate churn discrimination; fully deterministic/structural relationship with `phone_service` via Cramér's V magnitude.
- `streaming_tv`: weak univariate churn discrimination; near-deterministic relationship with other premium add-ons via Cramér's V magnitude.
- `streaming_movies`: weak univariate churn discrimination; near-deterministic relationship with other premium add-ons via Cramér's V magnitude.
- `device_protection`: comparably weak univariate churn discrimination to other premium add-ons; slightly stronger associations (than `online_backup`) with remaining premium add-ons and numeric features via Cramér's V and eta-squared magnitudes, favouring its removal over `online_backup`.
- `total_charges`: strong structural relationship with `tenure` via Spearman correlation due to being structurally derived from `tenure` and `monthly_charges`.

Modifies `churn_df` by removing identifier `customer_id`and redundant features `gender`, `phone_service`, `multiple_lines`, `streaming_tv`, `streaming_movies`, `device_protection`, and `total_charges`, in turn reducing the feature space from $20$ dimensions to $12$:

In [12]:
drop_cols = ["customer_id", "gender", "phone_service", "multiple_lines", "streaming_tv", "streaming_movies", 
             "device_protection", "total_charges"]

churn_df.drop(columns = drop_cols, inplace = True)

# confirms dimensions of feature space after removal:
print("Feature space dimensions after removing redundant features:", churn_df.shape[1] - 1)  # expected: 12

Feature space dimensions after removing redundant features: 12


Modifies `churn_df` by encoding target variable `churn` via `LabelEncoder`: 

In [13]:
label_encoder = LabelEncoder()      # initialises the LabelEncoder

churn_df["churn"] = label_encoder.fit_transform(churn_df["churn"])

# confirms encoded target classes:
print("Encoded target classes:", churn_df["churn"].unique())     # expected: [0 1]

Encoded target classes: [0 1]


Performs a $80/20$ split ($80\%$ training, $20\%$ testing) on `churn_df` with reproducible stratification for class proportion preservation:

In [14]:
# Extracts predictor and target variables into separate dataframes:
X_df = churn_df.iloc[:, :-1]
y_df = churn_df["churn"]

# performs 80/20 reproducible stratified split:
X_train, X_test, y_train, y_test = train_test_split(
    X_df,
    y_df,
    test_size = 0.2,
    stratify = y_df,
    random_state = 42)

# computes percentage proportions for encoded churn classes:
percentages_train = (y_train.value_counts(normalize = True) * 100).round(2)
percentages_test = (y_test.value_counts(normalize = True) * 100).round(2)

# confirms class proportions (~26.54% churn rate) are preserved:
print(f"Churn rate for training data: {percentages_train[1]}%")    # expected: ~26.54%
print(f"Churn rate for test data: {percentages_test[1]}%")         # expected: ~26.54%

Churn rate for training data: 26.58%
Churn rate for test data: 26.58%


One-hot encodes remaining categorical features (except`senior_citizen`), applies z-score standardisation to remaining numeric features, and creates new processed training and test data `X_train_processed` and `X_test_processed`:

In [15]:
# extracts column names of remaining categorical features:
cat_features = X_train.select_dtypes(include = ["object", "string"]).columns.tolist()

# extracts column names of remaining numerical features:
num_features = ["tenure", "monthly_charges"]

# initialises encoder by dropping first category of feature: 
cat_encoder = OneHotEncoder(drop = "first",    
                            handle_unknown = "ignore",       # prevents crashes   
                            sparse_output = False)           # returns NumPy array

# defines preprocessing transformers:
preprocessor = ColumnTransformer(
    transformers = [
        ("cat_encoder", cat_encoder, cat_features),           
        ("num_scaler", StandardScaler(), num_features)
    ],
    remainder = "passthrough",                  # retains unspecified features
    verbose_feature_names_out = False)          # removes suffixes

# forces preprocessor output to be dataframe:
preprocessor.set_output(transform = "pandas")

# fits encoder and scaler to training data:
X_train_processed = preprocessor.fit_transform(X_train)

# applies training parameters and encoder to testing data:
X_test_processed = preprocessor.transform(X_test)  

# extracts feature names defined in transformer:
original_features = preprocessor.get_feature_names_out()

# modifies names by replacing spaces with underscores and changing uppercase to lowercase:
updated_features = [name.replace(" ", "_").lower() for name in original_features]

# updates feature column names of training and test data:
X_train_processed.columns = updated_features
X_test_processed.columns = updated_features

# confirms total number of features and new feature names after encoding and scaling:
print("Total features after one-hot encoding:", len(X_train_processed.columns.tolist()))
print("Feature space after one-hot encoding:\n", X_train_processed.columns.tolist(), sep = "")

Total features after one-hot encoding: 19
Feature space after one-hot encoding:
['partner_yes', 'dependents_yes', 'internet_service_fiber_optic', 'internet_service_no', 'online_security_no_internet_service', 'online_security_yes', 'online_backup_no_internet_service', 'online_backup_yes', 'tech_support_no_internet_service', 'tech_support_yes', 'contract_one_year', 'contract_two_year', 'paperless_billing_yes', 'payment_method_credit_card_(automatic)', 'payment_method_electronic_check', 'payment_method_mailed_check', 'tenure', 'monthly_charges', 'senior_citizen']


*Note: One-hot encoding is selected because categorical features are nominal with no inherent ranking. Despite tree-based models being able to handle all category features generated through encoding, `drop = "first"` is used to avoid multicollinearity for the Logistic Regression baseline. Similarly, despite tree-based models being capable of effectively handling unstandardised features, z-score scaling is applied for SHAP-magnitude comparability purposes (Phase 6). The fit/transform split is safe from data leakage (`fit_transform` on `X_train`, `transform` on `X_test`).*

Exports the processed training and test data to file:

In [16]:
data = [X_train_processed, X_test_processed, y_train, y_test]
data_names = ["X_train_processed", "X_test_processed", "y_train", "y_test"]

for feature_data, names in zip(data, data_names):
    feature_data.to_csv(f"data/processed/{names}.csv", index = False)
    print(f"Export complete for {names}! Shape:", feature_data.shape) 

Export complete for X_train_processed! Shape: (5625, 19)
Export complete for X_test_processed! Shape: (1407, 19)
Export complete for y_train! Shape: (5625,)
Export complete for y_test! Shape: (1407,)


##### $\textbf{SQL Database Creation}$

Creates `telco_churn` relational database and populates it with the churn dataset after removing the $11$ rows containing blank entries:

In [17]:
os.makedirs("data/sql", exist_ok = True)     # creates a directory if it does not exist

# establishes connection to SQLite and creates 'telco_churn' database:
conn = sqlite3.connect("data/sql/telco_churn.db") 

# 'if_exists' argument ensures recreation of table each execution:
db_churn_df.to_sql("telco_churn", conn, if_exists = "replace", index = False)

# verifies row count using query:
row_verification = pd.read_sql("SELECT COUNT(*) AS row_count FROM telco_churn", conn)
print(row_verification)                      # expected: 7032

# closes connection:
conn.close()

   row_count
0       7032


##### $\textbf{Notebook Outputs}$

The outputs of this notebook are as follows: training and testing data `churn_X_train_processed`, `churn_X_test_processed`, `churn_y_train`, and `churn_y_test`, and `telco_churn.db` (database populated with `churn_df` after removal of $11$ rows). Both outputs are ready for use in subsequent phases of the workflow.